# TOFU SISA-LoRA Unlearning

SISA-inspired machine unlearning using per-shard LoRA adaptors on the TOFU dataset.

**Pipeline:**
1. Split 200 TOFU authors into k author-aligned shards
2. Train one LoRA adaptor per shard (independently, parallelizable via SLURM)
3. Merge all k LoRAs into one model (linear average / DARE / TIES)
4. Evaluate with standard TOFU unlearning metrics
5. Unlearn a shard by re-merging without it (no retraining)
6. Hyperparameter sweep over k, aggregation method, unlearn method

**Key fact:** forget10 authors are rows 3600–3999 (author IDs 180–199).  
With k=10, shard 9 = exactly forget10. With k=4, shard 3 ⊇ forget10.

## Section 0 — Config

In [ ]:
import os
os.environ["HF_HOME"] = "/storage2/jack/data/huggingface"

# Multi-model checkpoint roots (see submit_all_models.sh / submit_all_eval.sh)
MODELS = {
    "tinyllama": (
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "./checkpoints/TinyLlama-1.1B-Chat-v1.0",
    ),
    "phi2": ("microsoft/phi-2", "./checkpoints/phi-2"),
    "llama32_1b": (
        "meta-llama/Llama-3.2-1B-Instruct",
        "./checkpoints/Llama-3.2-1B-Instruct",
    ),
}
ACTIVE_MODEL = "tinyllama"  # switch for merge/eval in this notebook

CONFIG = {
    "k": 4,
    "seed": 42,
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"],
    "epochs": 3,
    "batch_size": 4,
    "grad_accum": 4,
    "lr": 2e-4,
    "max_length": 256,
    "train_parallel": True,
    "slurm_exclude": "sprint4",  # 1 GPU per job; scheduler picks sprint1/2/3
    "shard_to_forget": None,
}

CONFIG["model_name"], CONFIG["output_dir"] = MODELS[ACTIVE_MODEL]

if CONFIG["shard_to_forget"] is None:
    CONFIG["shard_to_forget"] = CONFIG["k"] - 1

assert 200 % CONFIG["k"] == 0, f"k={CONFIG['k']} must evenly divide 200"
print(f"k={CONFIG['k']} shards, {200 // CONFIG['k']} authors each")
print(f"Forget shard: {CONFIG['shard_to_forget']}")
print(f"Model: {CONFIG['model_name']}")

## Section 1 — Data Loading & Sharding

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load all TOFU splits we need
full_ds        = load_dataset("locuslab/TOFU", "full")["train"]
forget10_ds    = load_dataset("locuslab/TOFU", "forget10")["train"]
retain90_ds    = load_dataset("locuslab/TOFU", "retain90")["train"]
forget10_pert  = load_dataset("locuslab/TOFU", "forget10_perturbed")["train"]
retain_pert    = load_dataset("locuslab/TOFU", "retain_perturbed")["train"]
real_authors   = load_dataset("locuslab/TOFU", "real_authors")["train"]
world_facts    = load_dataset("locuslab/TOFU", "world_facts")["train"]

print(f"full: {len(full_ds)}, forget10: {len(forget10_ds)}, retain90: {len(retain90_ds)}")
print(f"real_authors: {len(real_authors)}, world_facts: {len(world_facts)}")

# Verify forget10 = last 20 authors (IDs 180-199)
forget10_qs = set(forget10_ds["question"])
forget_indices = [i for i, r in enumerate(full_ds) if r["question"] in forget10_qs]
forget_author_ids = sorted(set(i // 20 for i in forget_indices))
assert forget_author_ids == list(range(180, 200)), "Expected forget10 = authors 180-199"
print(f"\nVerified: forget10 authors are IDs {forget_author_ids[0]}–{forget_author_ids[-1]}")

In [ ]:
def get_author_shard(k, shard_id):
    """Author IDs for shard_id. Natural order — shard k-1 always = forget10 authors."""
    authors_per_shard = 200 // k
    start = shard_id * authors_per_shard
    return list(range(start, start + authors_per_shard))


def authors_to_dataset(author_ids, full_ds):
    """Select all 20 Q&As for each author and format as 'Question: ...\nAnswer: ...'."""
    indices = [r for a in author_ids for r in range(a * 20, a * 20 + 20)]
    ds = full_ds.select(indices)
    return ds.map(
        lambda ex: {"text": f"Question: {ex['question']}\nAnswer: {ex['answer']}"},
        remove_columns=["question", "answer"],
    )


k = CONFIG["k"]
shards = {i: get_author_shard(k, i) for i in range(k)}
shard_datasets = {i: authors_to_dataset(shards[i], full_ds) for i in range(k)}

# Sanity checks
all_author_ids = [a for shard in shards.values() for a in shard]
assert len(all_author_ids) == 200
assert len(set(all_author_ids)) == 200, "Authors appear in multiple shards!"

print("Shard breakdown:")
for i in range(k):
    overlap = set(shards[i]) & set(forget_author_ids)
    note = f" ← contains {len(overlap)} forget10 authors" if overlap else ""
    print(f"  shard {i}: authors {shards[i][0]}–{shards[i][-1]}, {len(shard_datasets[i])} Q&As{note}")

## Section 2 — LoRA Training

**Default (Option B):** SLURM array — **k parallel jobs, 1 GPU each** (uses all 4 GPUs on `sprint1` when k=4).

**Option A** (sequential): set `TRAIN_SEQUENTIAL = True` in the next cell — one GPU, trains shards back-to-back.

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer


def train_lora_for_shard(shard_id, dataset, cfg, output_dir):
    import json, os

    save_dir = os.path.join(output_dir, f"shard_{shard_id}")
    if os.path.exists(os.path.join(save_dir, "adapter_config.json")):
        print(f"  shard {shard_id}: already trained, skipping")
        return

    print(f"\n=== Training shard {shard_id} ({len(dataset)} samples) ===")

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"], trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        cfg["model_name"],
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False

    lora_cfg = LoraConfig(
        r=cfg["lora_rank"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["lora_target_modules"],
        bias="none",
        task_type="CAUSAL_LM",
        use_rslora=True,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    os.makedirs(save_dir, exist_ok=True)
    train_args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["grad_accum"],
        optim="paged_adamw_32bit",
        learning_rate=cfg["lr"],
        weight_decay=0.001,
        bf16=True,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=cfg["seed"],
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=train_args,
        dataset_text_field="text",
        max_seq_length=cfg["max_length"],
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()

    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    meta = {
        "shard_id": shard_id, "k": cfg["k"],
        "author_ids": shards[shard_id],
        "num_samples": len(dataset),
        "model_name": cfg["model_name"],
        "rank": cfg["lora_rank"],
    }
    with open(os.path.join(save_dir, "shard_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    # Free GPU memory before next shard
    del model, trainer
    torch.cuda.empty_cache()
    print(f"  Saved -> {save_dir}")


# --- Option A: sequential training (single GPU) ---
TRAIN_SEQUENTIAL = False  # True = train all shards in this notebook; False = SLURM parallel (next cell)

if TRAIN_SEQUENTIAL:
    for shard_id in range(CONFIG["k"]):
        train_lora_for_shard(shard_id, shard_datasets[shard_id], CONFIG, CONFIG["output_dir"])
    print("\nAll shards trained.")
else:
    print("Skipping in-notebook training — use parallel SLURM (next cell).")

In [ ]:
# --- Option B: SLURM array (k parallel jobs x 1 GPU on sprint1) ---
import subprocess
import re

PROJECT_DIR = os.path.abspath(os.path.join(CONFIG["output_dir"], ".."))


def submit_parallel_training(cfg):
    """1 GPU per shard; SLURM excludes sprint4 (see slurm_nodes.sh)."""
    k = cfg["k"]
    model = cfg["model_name"]
    cmd = ["bash", os.path.join(PROJECT_DIR, "submit_overnight.sh"), str(k), model]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_DIR)
    print(result.stdout.strip() or result.stderr.strip())
    m = re.search(r"Submitted batch job (\d+)", result.stdout + result.stderr)
    return int(m.group(1)) if m else None


def missing_shards(cfg):
    out = []
    for i in range(cfg["k"]):
        p = os.path.join(cfg["output_dir"], f"shard_{i}", "adapter_config.json")
        if not os.path.exists(p):
            out.append(i)
    return out


TRAINING_JOB_ID = None
if CONFIG.get("train_parallel", True) and not TRAIN_SEQUENTIAL:
    missing = missing_shards(CONFIG)
    if missing:
        print(f"Missing shards {missing} — submitting {CONFIG['k']} parallel GPU jobs...")
        TRAINING_JOB_ID = submit_parallel_training(CONFIG)
        if TRAINING_JOB_ID:
            print(f"Job ID: {TRAINING_JOB_ID}  |  Monitor: squeue -u $USER")
    else:
        print(f"All {CONFIG['k']} shard checkpoints exist — no training submitted.")
else:
    print("Parallel SLURM disabled (TRAIN_SEQUENTIAL=True or train_parallel=False).")

In [ ]:
# Wait for parallel training, then verify checkpoints
import time


def wait_for_shards(cfg, job_id=None, poll_sec=30, timeout_sec=12 * 3600):
    start = time.time()
    while time.time() - start < timeout_sec:
        missing = missing_shards(cfg)
        if not missing:
            return True
        if job_id is not None:
            q = subprocess.run(
                ["squeue", "-j", str(job_id), "-h"],
                capture_output=True, text=True,
            )
            if not q.stdout.strip() and missing:
                print(f"Job {job_id} finished but still missing shards: {missing}")
                return False
        print(f"Waiting for shards {missing} ... ({int(time.time() - start)}s)", flush=True)
        time.sleep(poll_sec)
    raise TimeoutError(f"Shard training did not finish within {timeout_sec}s")


for name, (model_name, output_dir) in MODELS.items():
    cfg_m = {**CONFIG, "model_name": model_name, "output_dir": output_dir}
    if TRAINING_JOB_ID is not None and name == ACTIVE_MODEL:
        wait_for_shards(cfg_m, job_id=TRAINING_JOB_ID)
    missing = missing_shards(cfg_m)
    if missing:
        print(f"[{name}] still missing shards: {missing} (wait for SLURM jobs)")
    else:
        print(f"[{name}] all {CONFIG['k']} shard checkpoints ready.")

missing_active = missing_shards(CONFIG)
if missing_active:
    raise RuntimeError(
        f"Active model '{ACTIVE_MODEL}' missing shards: {missing_active}. "
        "Wait for training or run: bash submit_all_models.sh"
    )

## Section 3 — LoRA Aggregation

Load all k LoRA adaptors into a single PEFT model and combine them. Merge logic
lives in `merge_lora.py` (shared with the SLURM eval path). See
`LoRA Merging_ Methods and Effectiveness.md` for the method survey.

| Method | `combination_type` | Result rank | Notes |
|---|---|---|---|
| Linear average | `linear` | r | Task Arithmetic baseline; only method with exact `subtract` unlearn |
| DARE + linear | `dare_linear` | r | Random drop + rescale, then sum; reduces redundancy |
| TIES | `ties` | r | Trim → elect sign → merge; good for conflicting updates |
| DARE + TIES | `dare_ties` | r | Research-doc default for 3+ adapters |
| Magnitude prune | `magnitude_prune` | r | Keep top-density by magnitude, then sum |
| Concatenate | `cat` | k·r | Exact compose; rank grows with k |
| TIES / DARE-TIES SVD | `ties_svd`, `dare_ties_svd` | r | KnOTS-style: merge then compress back to rank r (opt-in) |

Unlearn (forget one shard): `remerge_*` (re-merge excluding the shard, works for
all methods) or `subtract_linear` (task-vector subtraction, linear only).

In [ ]:
from peft import PeftModel


def load_all_adapters(cfg, output_dir):
    """Load base model and attach all k LoRA adaptors as named adapters."""
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"], trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

    base = AutoModelForCausalLM.from_pretrained(
        cfg["model_name"],
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

    shard_0_dir = os.path.join(output_dir, "shard_0")
    model = PeftModel.from_pretrained(base, shard_0_dir, adapter_name="shard_0")

    for i in range(1, cfg["k"]):
        shard_dir = os.path.join(output_dir, f"shard_{i}")
        model.load_adapter(shard_dir, adapter_name=f"shard_{i}")

    print(f"Loaded {cfg['k']} adapters: {list(model.peft_config.keys())}")
    return model, tokenizer


# Merge logic is shared with the SLURM path (eval_tofu.py) via merge_lora.py.
from merge_lora import DEFAULT_MERGE_METHODS, MERGE_METHODS, create_unlearn_subtract, merge_shards

peft_model, tokenizer = load_all_adapters(CONFIG, CONFIG["output_dir"])

# Create one merged adapter per method. Each call returns the peft adapter name.
merged_names = {}
for method in DEFAULT_MERGE_METHODS:
    name = merge_shards(peft_model, CONFIG["k"], method)
    merged_names[method] = name
    print(f"Created adapter: {name}")

print("\nMethods available:", sorted(MERGE_METHODS))
print("All adapters:", list(peft_model.peft_config.keys()))

## Section 4 — Evaluation

Standard TOFU unlearning metrics (Maini et al. 2024).

**Forget Quality**: does the model forget?
- `forget_ppl`: perplexity on forget shard (↑ after unlearning = better)
- `forget_rouge`: ROUGE-L of generated answers on forget set (↓ = better)
- `truth_ratio`: P(correct) / mean P(perturbed) on forget set (↓ toward 1 = better)
- `ks_pval`: KS-test p-value vs. base model per-sample log-probs (↑ = more like a model that never saw the data)

**Model Utility**: does non-forgotten knowledge survive?
- `retain_ppl`: perplexity on retain set (stable = good)
- `retain_rouge`, `real_authors_rouge`, `world_facts_rouge`: ROUGE-L scores (stable = good)
- `model_utility`: harmonic mean of the three ROUGE scores above

In [ ]:
import math
import numpy as np
import torch
from scipy.stats import ks_2samp
from evaluate import load as load_metric

rouge_metric = load_metric("rouge")


def get_perplexity(model, tokenizer, texts, batch_size=8, max_length=256):
    """Average per-token NLL (= log perplexity) for a list of strings."""
    model.eval()
    device = next(model.parameters()).device
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=max_length,
            ).to(device)
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100
            out = model(**enc, labels=labels)
            n_tok = (labels != -100).sum().item()
            total_nll += out.loss.item() * n_tok
            total_tokens += n_tok
    return math.exp(total_nll / total_tokens)


def get_per_sample_logprobs(model, tokenizer, texts, max_length=256):
    """Per-sample mean log-prob; used for KS-test."""
    model.eval()
    device = next(model.parameters()).device
    logprobs = []
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(
                text, return_tensors="pt", truncation=True, max_length=max_length
            ).to(device)
            labels = enc["input_ids"].clone()
            out = model(**enc, labels=labels)
            logprobs.append(-out.loss.item())  # loss = mean NLL, so -loss = mean log-prob
    return np.array(logprobs)


def get_rouge(model, tokenizer, questions, gold_answers, max_new_tokens=100):
    """ROUGE-L of greedy-decoded answers vs. gold."""
    model.eval()
    device = next(model.parameters()).device
    preds = []
    with torch.no_grad():
        for q in questions:
            prompt = f"Question: {q}\nAnswer:"
            enc = tokenizer(prompt, return_tensors="pt").to(device)
            ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_ids = ids[0][enc["input_ids"].shape[1]:]
            preds.append(tokenizer.decode(new_ids, skip_special_tokens=True).strip())
    result = rouge_metric.compute(predictions=preds, references=gold_answers, use_stemmer=True)
    return result["rougeL"]  # F1


def get_truth_ratio(model, tokenizer, perturbed_ds, max_length=256):
    """Mean ratio P(true_answer) / mean_P(perturbed_answers) per question.
    Computed as exp(mean_logprob_true - mean_logprob_perturbed).
    """
    model.eval()
    device = next(model.parameters()).device
    ratios = []
    with torch.no_grad():
        for row in perturbed_ds:
            q = row["question"]
            true_text = f"Question: {q}\nAnswer: {row['answer']}"
            perturbed_texts = [f"Question: {q}\nAnswer: {p}" for p in row["perturbed_answer"]]

            def mean_logprob(text):
                enc = tokenizer(
                    text, return_tensors="pt", truncation=True, max_length=max_length
                ).to(device)
                out = model(**enc, labels=enc["input_ids"].clone())
                return -out.loss.item()

            lp_true = mean_logprob(true_text)
            lp_pert = np.mean([mean_logprob(t) for t in perturbed_texts])
            ratios.append(math.exp(lp_true - lp_pert))
    return float(np.mean(ratios))


def harmonic_mean(values):
    return len(values) / sum(1.0 / v for v in values if v > 0)


def evaluate_model(model, tokenizer, label, forget_shard_id,
                   full_ds, shards, forget10_pert, retain_pert,
                   real_authors, world_facts,
                   base_logprobs=None):
    """Compute all TOFU unlearning metrics for the currently active adapter."""

    forget_authors = shards[forget_shard_id]
    forget_indices = [r for a in forget_authors for r in range(a * 20, a * 20 + 20)]
    forget_ds = full_ds.select(forget_indices)
    retain_indices = [i for i in range(len(full_ds)) if i not in set(forget_indices)]
    # Subsample retain for speed (500 random samples)
    rng = np.random.default_rng(0)
    retain_sample = rng.choice(retain_indices, size=min(500, len(retain_indices)), replace=False).tolist()
    retain_ds = full_ds.select(retain_sample)

    forget_texts  = [f"Question: {r['question']}\nAnswer: {r['answer']}" for r in forget_ds]
    retain_texts  = [f"Question: {r['question']}\nAnswer: {r['answer']}" for r in retain_ds]
    real_texts    = [f"Question: {r['question']}\nAnswer: {r['answer']}" for r in real_authors]
    world_texts   = [f"Question: {r['question']}\nAnswer: {r['answer']}" for r in world_facts]

    forget_ppl  = get_perplexity(model, tokenizer, forget_texts)
    retain_ppl  = get_perplexity(model, tokenizer, retain_texts)

    forget_rouge = get_rouge(model, tokenizer,
                             forget_ds["question"], forget_ds["answer"])
    retain_rouge = get_rouge(model, tokenizer,
                             retain_ds["question"], retain_ds["answer"])
    real_rouge   = get_rouge(model, tokenizer,
                             real_authors["question"], real_authors["answer"])
    world_rouge  = get_rouge(model, tokenizer,
                             world_facts["question"], world_facts["answer"])

    # Truth ratio uses the perturbed split; if forget shard != forget10 authors,
    # filter forget10_perturbed to the questions in this shard
    forget_qs = set(forget_ds["question"])
    pert_subset = forget10_pert.filter(lambda r: r["question"] in forget_qs)
    truth_ratio = get_truth_ratio(model, tokenizer, pert_subset) if len(pert_subset) > 0 else float("nan")

    # KS-test: compare log-prob distribution on forget set to base model
    model_logprobs = get_per_sample_logprobs(model, tokenizer, forget_texts)
    if base_logprobs is not None:
        ks_stat, ks_pval = ks_2samp(model_logprobs, base_logprobs)
    else:
        ks_pval = float("nan")

    model_utility = harmonic_mean([retain_rouge, real_rouge, world_rouge])

    return {
        "label":          label,
        "forget_ppl":     round(forget_ppl, 2),
        "retain_ppl":     round(retain_ppl, 2),
        "forget_rouge":   round(forget_rouge, 4),
        "retain_rouge":   round(retain_rouge, 4),
        "real_rouge":     round(real_rouge, 4),
        "world_rouge":    round(world_rouge, 4),
        "truth_ratio":    round(truth_ratio, 4),
        "ks_pval":        round(ks_pval, 4) if not math.isnan(ks_pval) else float("nan"),
        "model_utility":  round(model_utility, 4),
    }

In [ ]:
# --- Get base model log-probs on forget set (used as gold-model proxy for KS-test) ---
peft_model.set_adapter("shard_0")  # temporarily use any adapter
# Load the base model separately (disable all LoRA) to get base log-probs
base_model_only = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
forget_shard_id = CONFIG["shard_to_forget"]
forget_authors  = shards[forget_shard_id]
forget_indices  = [r for a in forget_authors for r in range(a * 20, a * 20 + 20)]
forget_ds       = full_ds.select(forget_indices)
forget_texts    = [f"Question: {r['question']}\nAnswer: {r['answer']}" for r in forget_ds]

base_logprobs = get_per_sample_logprobs(base_model_only, tokenizer, forget_texts)
del base_model_only; torch.cuda.empty_cache()
print(f"Base model mean log-prob on forget set: {base_logprobs.mean():.3f}")

In [ ]:
results = []

# Evaluate each shard adaptor individually (one nbconvert timeout budget per cell)
for i in range(CONFIG["k"]):
    print(f"Evaluating shard_{i} ...", flush=True)
    peft_model.set_adapter(f"shard_{i}")
    row = evaluate_model(
        peft_model, tokenizer, f"shard_{i}_only",
        forget_shard_id=CONFIG["shard_to_forget"],
        full_ds=full_ds, shards=shards,
        forget10_pert=forget10_pert, retain_pert=retain_pert,
        real_authors=real_authors, world_facts=world_facts,
        base_logprobs=base_logprobs,
    )
    results.append(row)
    print(row, flush=True)

In [ ]:
# Evaluate each merged variant (separate cell — own timeout budget)
for combo, name in merged_names.items():
    print(f"Evaluating merged_{combo} ...", flush=True)
    peft_model.set_adapter(name)
    row = evaluate_model(
        peft_model, tokenizer, f"merged_{combo}",
        forget_shard_id=CONFIG["shard_to_forget"],
        full_ds=full_ds, shards=shards,
        forget10_pert=forget10_pert, retain_pert=retain_pert,
        real_authors=real_authors, world_facts=world_facts,
        base_logprobs=base_logprobs,
    )
    results.append(row)
    print(row, flush=True)

df_results = pd.DataFrame(results).set_index("label")
print(df_results.to_string())

## Section 5 — Unlearning Demo

Forget shard `CONFIG["shard_to_forget"]` from the merged model (no retraining).

- **Method 1 (re-merge)**: works for all aggregation methods; exact.
- **Method 2 (delta subtraction)**: linear-only; faster — no extra merge call needed if weights saved.

In [ ]:
forget_id = CONFIG["shard_to_forget"]

# Method 1: re-merge without the forget shard (works for all combination types)
for method in DEFAULT_MERGE_METHODS:
    name = merge_shards(peft_model, CONFIG["k"], method, exclude_shard=forget_id)
    print(f"Created unlearned adapter: {name}")

# Method 2: task-vector subtraction (subtract the forget shard's delta).
# Implemented with `cat` in merge_lora because PEFT's linear path can't take
# the negative weight this needs. W_unlearn ~= W_merged - (1/k)*dW_forget.
sub_name = create_unlearn_subtract(peft_model, CONFIG["k"], forget_id)
print(f"Created adapter: {sub_name}")

In [ ]:
# Evaluate all unlearned variants
unlearn_adapters = [
    (f"merged_linear_no{forget_id}",      "remerge_linear"),
    (f"merged_dare_linear_no{forget_id}", "remerge_dare"),
    (f"merged_ties_no{forget_id}",        "remerge_ties"),
    ("unlearn_subtract",                  "subtract_linear"),
]

unlearn_results = []
for adapter_name, label in unlearn_adapters:
    peft_model.set_adapter(adapter_name)
    row = evaluate_model(
        peft_model, tokenizer, label,
        forget_shard_id=CONFIG["shard_to_forget"],
        full_ds=full_ds, shards=shards,
        forget10_pert=forget10_pert, retain_pert=retain_pert,
        real_authors=real_authors, world_facts=world_facts,
        base_logprobs=base_logprobs,
    )
    unlearn_results.append(row)

df_unlearn = pd.DataFrame(unlearn_results).set_index("label")

# Combine into one comparison table
df_all = pd.concat([
    df_results.loc[[f"merged_linear"]].rename(index={"merged_linear": "merged (before)"}),
    df_unlearn,
])
print("=== Unlearning Results ===")
print(df_all[["forget_ppl", "retain_ppl", "forget_rouge", "retain_rouge", "truth_ratio", "ks_pval", "model_utility"]].to_string())
print("\nGoal: forget_ppl↑  forget_rouge↓  truth_ratio→1  ks_pval↑  retain/model_utility stable")

In [ ]:
# Quick qualitative check: generate answers for a forget-set question before and after
import textwrap

sample_q = forget_ds[0]["question"]
sample_a = forget_ds[0]["answer"]
prompt   = f"Question: {sample_q}\nAnswer:"
enc      = tokenizer(prompt, return_tensors="pt").to(next(peft_model.parameters()).device)

print(f"Question: {sample_q}")
print(f"Gold:     {sample_a}\n")

for adapter_name, label in [
    (merged_names["linear"], "merged (before unlearn)"),
    (f"merged_linear_no{forget_id}", "re-merge (after unlearn)"),
    ("unlearn_subtract", "subtract (after unlearn)"),
]:
    peft_model.set_adapter(adapter_name)
    with torch.no_grad():
        ids = peft_model.generate(**enc, max_new_tokens=80, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(ids[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print(f"[{label}]\n{textwrap.fill(gen, 80)}\n")

## Section 6 — Hyperparameter Sweep

Sweep over k, aggregation method, and unlearning method.  
Each (k, shard_id) combination is submitted as a SLURM array task.  
Results are collected by re-running the eval cells after all jobs finish.

**Constraint**: forget10 authors are IDs 180–199 (the last 20). The last shard contains
all 20 forget10 authors only when k ≤ 10. For k > 10 the forget10 block spans multiple shards.

| k | authors/shard | forget10 in shard k-1? | LoRA runs |
|---|---|---|---|
| 4 | 50 | ✅ all 20 (+ 30 others) | 4 |
| 5 | 40 | ✅ all 20 (+ 20 others) | 5 |
| 10 | 20 | ✅ exactly forget10 | 10 |
| 20 | 10 | ❌ split across shards 18 & 19 | 20 |

In [ ]:
SWEEP_CONFIG = {
    # k must divide 200 evenly; only k<=10 keeps all forget10 authors in one shard
    "k_values":        [4, 5, 10],
    "aggregations":    ["linear", "dare_linear", "ties"],
    "unlearn_methods": ["remerge", "subtract"],
    "lora_ranks":      [8],   # add [4, 8, 16] for a rank sweep
    "base_config":     CONFIG,
}


def submit_sweep(sweep_cfg, partition="all", time="02:00:00"):
    """Submit one SLURM array job per k value."""
    base = sweep_cfg["base_config"]
    for k in sweep_cfg["k_values"]:
        sweep_dir = f"{base['output_dir']}_k{k}"
        sweep_cfg_k = {**base, "k": k, "output_dir": sweep_dir}
        submit_parallel_training(sweep_cfg_k)
        print(f"Submitted k={k} -> {sweep_dir}")


# submit_sweep(SWEEP_CONFIG)  # uncomment to run the full sweep

In [ ]:
# Collect sweep results (run after all jobs finish)
# For each k: load adapters, merge with each aggregation, unlearn, evaluate.

sweep_rows = []

for k in SWEEP_CONFIG["k_values"]:
    sweep_dir = f"{CONFIG['output_dir']}_k{k}"
    # Check if checkpoints exist
    if not all(os.path.exists(f"{sweep_dir}/shard_{i}/adapter_config.json") for i in range(k)):
        print(f"k={k}: checkpoints not ready, skipping")
        continue

    sweep_k_cfg = {**CONFIG, "k": k, "output_dir": sweep_dir}
    shards_k    = {i: get_author_shard(k, i) for i in range(k)}
    model_k, tok_k = load_all_adapters(sweep_k_cfg, sweep_dir)

    forget_id_k = k - 1  # always the last shard

    for combo in SWEEP_CONFIG["aggregations"]:
        merged_k = merge_shards(model_k, k, combo)
        model_k.set_adapter(merged_k)
        before = evaluate_model(
            model_k, tok_k, f"k={k}|{combo}|before",
            forget_shard_id=forget_id_k, full_ds=full_ds, shards=shards_k,
            forget10_pert=forget10_pert, retain_pert=retain_pert,
            real_authors=real_authors, world_facts=world_facts,
            base_logprobs=base_logprobs,
        )
        before["k"] = k; before["aggregation"] = combo; before["unlearn"] = "none"
        sweep_rows.append(before)

        # Unlearn via re-merge (works for all combo types)
        unlearned_name = merge_shards(model_k, k, combo, exclude_shard=forget_id_k)
        model_k.set_adapter(unlearned_name)
        after = evaluate_model(
            model_k, tok_k, f"k={k}|{combo}|remerge",
            forget_shard_id=forget_id_k, full_ds=full_ds, shards=shards_k,
            forget10_pert=forget10_pert, retain_pert=retain_pert,
            real_authors=real_authors, world_facts=world_facts,
            base_logprobs=base_logprobs,
        )
        after["k"] = k; after["aggregation"] = combo; after["unlearn"] = "remerge"
        sweep_rows.append(after)

    del model_k; torch.cuda.empty_cache()

if sweep_rows:
    df_sweep = pd.DataFrame(sweep_rows)
    print(df_sweep[["k", "aggregation", "unlearn", "forget_ppl", "retain_ppl",
                    "forget_rouge", "truth_ratio", "ks_pval", "model_utility"]].to_string())
else:
    print("No sweep results yet — run submit_sweep() and wait for SLURM jobs to finish.")